1. Environment Setup and Data Ingestion

This section configures the system path to access local source modules and initializes the data pipeline. It executes two primary operations: loading the pre-processed dataset from the local cache and establishing the validation protocol. The validation split is generated using a specific fixed seed (seed=42) to replicate the exact data partitioning used in the previous tree-based experiments. This guarantees a strictly identical validation set, enabling a direct and fair comparison of the Exact Match Ratio (EMR) metrics between models.

In [1]:
import os
import sys

# Ensures Python can locate the 'src' directory
sys.path.insert(0, os.path.abspath('..'))

from src import data
from src import experiment as E

# 1. Attempt to load the dataset
print("Loading data...")
try:
    d = data.load()
    print("✅ Success! Data loaded successfully.")
    print(f"Training dimensions (CRM): {d['X_train'].shape}")
    print(f"Test dimensions (CRM): {d['X_test'].shape}")
    
    # 2. Verify validation split generation
    setup_data = E.setup(seed=42)
    print(f"✅ Success! {len(setup_data['tr'])} rows for training, {len(setup_data['va'])} for internal validation.")

except Exception as error:
    print(f"❌ Error loading data: {error}")

Loading data...
✅ Success! Data loaded successfully.
Training dimensions (CRM): (187442, 745)
Test dimensions (CRM): (97100, 745)
✅ Success! 151057 rows for training, 36385 for internal validation.


2. Data Deduplication and Sparse Matrix Transformation

This section prepares the feature matrices for the linear model. The training data is first deduplicated to isolate unique CRM configurations, pairing them with their consensus billing targets. This structural step mitigates label noise and prevents highly frequent configurations from disproportionately influencing the model's weights. Subsequently, both training and validation sets are converted into Compressed Sparse Row (CSR) format. This transformation minimizes memory footprint and maximizes the computational efficiency of the sparse-aware solver utilized in the logistic regression.

In [2]:
from scipy.sparse import csr_matrix
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np

# 1. Load all data at once via setup (internally handles data.load)
d = E.setup(seed=42)

# 2. Extract training and validation indices
tr, va = d['tr'], d['va']

# 3. Deduplicate configurations (d['Y_cons'] is now accessible)
Xa_dense, Ya, cnt = E.dedup_configs(d['X_train'][tr], d['Y_cons'][tr])

print(f"Original training rows: {len(tr)}")
print(f"Unique configurations after deduplication: {Xa_dense.shape[0]}")

# 4. Convert to sparse matrices
Xa = csr_matrix(Xa_dense)
X_val = csr_matrix(d['X_train'][va])
Yv = d['Y_cons'][va] # Ground truth validation targets

print("Sparse format conversion completed successfully!")

Original training rows: 151057
Unique configurations after deduplication: 53947
Sparse format conversion completed successfully!


3. Correlation-Based Chain Ordering

A ClassifierChain model predicts labels sequentially, meaning the order of variables matters. This section determines the optimal sequence by grouping highly correlated labels together:

Graph Construction: It loads the pre-calculated label correlations (near_duplicate_bil_pairs.csv) and filters for relationships with a Jaccard similarity of 0.90 or higher.

Cluster Extraction: Using the networkx library, it builds an undirected graph where nodes are the 731 target labels and edges are the strong correlations. It then extracts the connected components (clusters of dependent labels).

Sorting Strategy: It orders the clusters so that groups containing the most frequent labels appear first. Within each cluster, the labels are also sorted by frequency in descending order.
Formatting: The final sorted list of label names is converted into the integer indices required by the scikit-learn API.

This ensures the model predicts the most common, independent labels first, and predicts highly correlated labels sequentially to capture their joint distribution.

In [4]:
import pandas as pd
df_pairs = pd.read_csv('../data/derived/near_duplicate_bil_pairs.csv')
print(df_pairs.columns)

Index(['col_a', 'col_b', 'jaccard', 'pct_rows_differing'], dtype='object')


In [5]:
import pandas as pd
import numpy as np
import networkx as nx
from scipy.sparse import csr_matrix
from src import experiment as E

# ==========================================
# STEP 1: Prepare the Data (same as Tier A)
# ==========================================
print("Loading and preparing data...")
d = E.setup(seed=42)
tr, va = d['tr'], d['va']

# Deduplicate configurations and convert to sparse format
Xa_dense, Ya, cnt = E.dedup_configs(d['X_train'][tr], d['Y_cons'][tr])
Xa = csr_matrix(Xa_dense)
X_val = csr_matrix(d['X_train'][va])
Yv = d['Y_cons'][va]

bil_cols_list = list(d['bil_cols'])
print(f"✅ Data ready! Sparse matrix shape: {Xa.shape}")

# Calculate the frequency (marginal probability) of each label in the training set
label_freqs = Ya.mean(axis=0)
freq_dict = {label: freq for label, freq in zip(bil_cols_list, label_freqs)}

# ==========================================
# STEP 2: Read Correlations and Order the Chain
# ==========================================
print("\nBuilding Classifier Chain order based on clusters...")

# 1. Load highly correlated pairs
df_pairs = pd.read_csv('../data/derived/near_duplicate_bil_pairs.csv')

# Filter only very strong correlations (Jaccard >= 0.90)
threshold = 0.90
strong_pairs = df_pairs[df_pairs['jaccard'] >= threshold]

# 2. Use graphs to discover label "Clusters" (Communities)
G = nx.Graph()
G.add_nodes_from(bil_cols_list) # Add all 731 labels as nodes

for _, row in strong_pairs.iterrows():
    # UPDATED: Using 'col_a' and 'col_b'
    G.add_edge(row['col_a'], row['col_b'], weight=row['jaccard'])

# Find connected components (our BIL clusters)
clusters = list(nx.connected_components(G))
print(f"Found {len(clusters)} clusters (including isolated single labels).")

# 3. Sort the clusters and the labels within them
custom_order_labels = []

# Sort clusters by the HIGHEST frequency label they contain
clusters_sorted = sorted(
    clusters, 
    key=lambda cluster: max(freq_dict[node] for node in cluster), 
    reverse=True
)

for cluster in clusters_sorted:
    # Within each cluster, sort from most frequent to least frequent
    nodes_sorted = sorted(list(cluster), key=lambda node: freq_dict[node], reverse=True)
    custom_order_labels.extend(nodes_sorted)

# 4. Convert BIL column names into integer indices required by Scikit-Learn
chain_order_indices = [bil_cols_list.index(col) for col in custom_order_labels]

print(f"✅ Chain order generated successfully! (Length: {len(chain_order_indices)})")
print(f"First 10 labels in the chain (anchors): {custom_order_labels[:10]}")

Loading and preparing data...
✅ Data ready! Sparse matrix shape: (53947, 745)

Building Classifier Chain order based on clusters...
Found 724 clusters (including isolated single labels).
✅ Chain order generated successfully! (Length: 731)
First 10 labels in the chain (anchors): ['BIL_BUSINESS_LINE_ORIG_Mobile', 'BIL_SUBSCRIBER_TYPE_ORIG_Prepaid', 'BIL_3748_PACK', 'BIL_3933_PACK', 'BIL_3934_PACK', 'BIL_3921_PACK', 'BIL_3927_PACK', 'BIL_3922_PACK', 'BIL_SUBSCRIBER_STATUS_ORIG_Active', 'BIL_3935_PACK']


4. Classifier Chain Training and Evaluation

The code configures an L1-penalized Logistic Regression wrapped in a ClassifierChain using the previously defined cluster-based order. Feature name warnings are temporarily suppressed because the chain dynamically adds past predictions as new input features during the sequential training of the 731 models.

During inference, the model generates probability predictions for the validation set. The scikit-learn implementation automatically realigns the output columns back to their original order. These probabilities are binarized using a 0.5 threshold, followed by the application of Sprint 1 deterministic additive rules to enforce known structural mappings. The final discrete predictions are evaluated against the ground truth to compute the Exact Match Ratio (EMR) and the classification report.

In [6]:
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegression
from IPython.display import display
from src import metrics as M
from src import data
import warnings

# Ignorar o aviso do scikit-learn sobre feature names (já que estamos a adicionar colunas novas na matriz)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

    print("Initializing Classifier Chain with L1 Logistic Regression...")
    base_lr = LogisticRegression(
        penalty='l1', 
        solver='liblinear', 
        C=1.0, 
        random_state=42
    )

    # Criar a Chain usando a ordem inteligente que acabaste de gerar
    chain_lr = ClassifierChain(base_lr, order=chain_order_indices, random_state=42)

    # Treinar (Atenção: vai demorar uns minutos porque é sequencial!)
    with E.Timer() as t:
        print("Training 731 Sequential Models... grab a coffee ☕")
        chain_lr.fit(Xa, Ya)
    print(f"✅ Training completed in {t.s:.0f} seconds.")

    # Prever probabilidades
    print("\nPredicting on validation set...")
    P_val = chain_lr.predict_proba(X_val)
    
    # O scikit-learn reordena automaticamente as colunas de volta para a ordem original (0 a 730),
    # por isso não precisamos de reordenar o P_val manualmente.
    pred_val = (P_val >= 0.5).astype(np.uint8)

    # Aplicar as Regras Aditivas do Sprint 1
    print("Applying Sprint 1 additive rules...")
    pred_val_final = data.apply_additive_rules(
        pred_val, 
        d['X_train'][va], 
        d['crm_cols'], 
        d['bil_cols']
    )

    # Avaliar o EMR
    print("\n🎯 RESULTS: CLASSIFIER CHAIN (L1 LR + RULES) 🎯")
    val_summary, val_cm, val_f1 = M.multilabel_report(Yv, pred_val_final, P_val)
    display(val_summary.to_frame('Chain LR Validation').style.format('{:.5f}'))

Initializing Classifier Chain with L1 Logistic Regression...
Training 731 Sequential Models... grab a coffee ☕
✅ Training completed in 1199 seconds.

Predicting on validation set...
Applying Sprint 1 additive rules...

🎯 RESULTS: CLASSIFIER CHAIN (L1 LR + RULES) 🎯


,Chain LR Validation
EMR,0.86816
bit accuracy,0.99839
Hamming loss,0.00161
F1 micro,0.96014
F1 samples,0.97042
F1 macro,0.34853
median per-label F1,0.10714
precision micro,0.98106
recall micro,0.94010
ROC-AUC micro,0.97785


5. Experiment Tracking and Result Logging

The code structures the experiment's metadata, structural configuration, and validation metrics (such as EMR and Hamming loss) into a dictionary record. It loads the central experiment tracker CSV, filters out any existing entries with the same experiment ID to prevent duplicates, and appends the new result. The updated dataframe is then saved back to disk, maintaining a persistent and standardized log of the model's performance for team comparison.

In [7]:
import pandas as pd
from datetime import datetime

# Estruturar o registo do modelo
meu_resultado_chain = {
    'exp_id': 'E-TierB-Chain-LR',
    'pipeline': 'Classifier Chain (Cluster Ordered) + L1 LR + Additive Rules',
    'family': 'ClassifierChain_LogisticRegression',
    'params': 'penalty=l1, solver=liblinear, order=correlation_clusters',
    'emr': val_summary['EMR'],
    'hamming_loss': val_summary['Hamming loss'],
    'train_time_s': t.s,
    'notes': 'EMR 86.81%. Median F1 saltou de 0.090 para 0.107 face à baseline OvR! A ordenação por clusters ajudou imenso os labels raros, à custa de 20 min de treino.'
}

# Carregar, atualizar e guardar
tracker_path = '../data/derived/sprint2_experiment_results.csv'
res = pd.read_csv(tracker_path)
res = res[res['exp_id'] != 'E-TierB-Chain-LR'] # Evitar duplicados
res = pd.concat([res, pd.DataFrame([meu_resultado_chain])], ignore_index=True)
res.to_csv(tracker_path, index=False)

print("✅ Modelo Classifier Chain adicionado com sucesso ao tracker da equipa!")

✅ Modelo Classifier Chain adicionado com sucesso ao tracker da equipa!
